In [12]:
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Tanvi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Tanvi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
import yt_dlp

In [3]:
link = "https://youtu.be/eR3IsKVLorg?si=29eoHdMYjn0tL-V0"
ydl_opts = {}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(link, download=False)

[youtube] Extracting URL: https://youtu.be/eR3IsKVLorg?si=29eoHdMYjn0tL-V0
[youtube] eR3IsKVLorg: Downloading webpage


[youtube] eR3IsKVLorg: Downloading android vr player API JSON


In [4]:
print("Title :", info['title'])
print("Views :", info['view_count'])
print("Duration :", info['duration'])
print("Description :", info['description'])
print("Ratings :", info['average_rating'])

Title : What You Know That AI Doesn’t | Priyanka Vergadia | TED
Views : 26638
Duration : 560
Description : AI is good at seeing patterns, but it’s humans who figure out what to do next, says technologist Priyanka Vergadia. She shares three stories of human excellence sparked by AI insights and offers a pathway to identify and cultivate your irreplaceable qualities, turning the AI revolution from a threat into an opportunity. (Recorded at TEDNext 2025 on November 10, 2025)

Join us in person at a TED conference: https://tedtalks.social/events
Become a TED Member to support our mission: https://ted.com/membership
Subscribe to a TED newsletter: https://ted.com/newsletters

Follow TED! 
X: https://www.twitter.com/TEDTalks
Instagram: https://www.instagram.com/ted
Facebook: https://facebook.com/TED
LinkedIn: https://www.linkedin.com/company/ted-conferences
TikTok: https://www.tiktok.com/@tedtoks

The TED Talks channel features talks, performances and original series from the world's leading 

# Downloading Youtube Video

In [5]:
ydl_opts = {'outtmpl': '%(title)s.%(ext)s'}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([link])
print("Download completed!!")

[youtube] Extracting URL: https://youtu.be/eR3IsKVLorg?si=29eoHdMYjn0tL-V0
[youtube] eR3IsKVLorg: Downloading webpage


[youtube] eR3IsKVLorg: Downloading android vr player API JSON


[info] eR3IsKVLorg: Downloading 1 format(s): 18
[download] What You Know That AI Doesn’t ｜ Priyanka Vergadia ｜ TED.mp4 has already been downloaded
[download] 100% of   23.59MiB
Download completed!!


In [ ]:
try:
    transcript = get_transcript(video_id)
except:
    print("Transcript not available for this video")
    exit()

Transcript not available for this video


: 

In [1]:
pip install yt-dlp youtube-transcript-api transformers sentence-transformers scikit-learn nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
def get_video_id(url):
    if "youtu.be/" in url:
        video_id = url.split("youtu.be/")[-1].split("?")[0]
    elif "v=" in url:
        video_id = url.split("v=")[-1].split("&")[0]
    elif "/watch/" in url:
        video_id = url.split("/watch/")[-1].split("?")[0]
    else:
        video_id = url
    return video_id

In [5]:
def get_transcript(video_id):
    api = YouTubeTranscriptApi()
    preferred_languages = ['en', 'en-US', 'en-GB', 'en-IN']
    try:
        transcript_data = api.fetch(video_id, languages=preferred_languages)
    except Exception as first_err:
        try:
            transcripts = api.list(video_id)
            try:
                transcript = transcripts.find_transcript(preferred_languages)
            except Exception:
                transcript = transcripts.find_generated_transcript(preferred_languages)

            if transcript.language_code not in preferred_languages:
                transcript = transcript.translate('en')

            transcript_data = transcript.fetch()
        except Exception as second_err:
            raise RuntimeError(f"Could not fetch transcript for {video_id}: {first_err} | {second_err}")

    if hasattr(transcript_data, 'snippets'):
        text = " ".join([snippet.text for snippet in transcript_data.snippets])
    else:
        text = " ".join([item.get('text', '') if isinstance(item, dict) else str(item) for item in transcript_data])
    return text

# Summary

In [6]:
import re


def summarize_text(text, max_sentences=8):
    if not text or not text.strip():
        return ""

    content = re.sub(r"\s+", " ", text.replace("\n", " ")).strip()
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', content) if s.strip()]
    if len(sentences) <= max_sentences:
        return " ".join(sentences)

    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(sentences, convert_to_tensor=True)

    centroid = np.mean(embeddings.cpu().numpy(), axis=0)
    sims = cosine_similarity([centroid], embeddings.cpu().numpy())[0]

    ranked_idxs = np.argsort(sims)[::-1][:max_sentences]
    ranked_idxs_sorted = sorted(ranked_idxs)

    summary_sentences = [sentences[i] for i in ranked_idxs_sorted]
    return " ".join(summary_sentences)


# Key point extraction

In [7]:
def extract_key_points(text, num_points=5):
    sentences = [s.strip() for s in text.replace("\n", " ").split('.') if s.strip()]

    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(sentences)

    scores = cosine_similarity(embeddings, embeddings.mean(axis=0).reshape(1, -1))

    ranked = sorted([(scores[i][0], s) for i, s in enumerate(sentences)], reverse=True)

    return [s for _, s in ranked[:num_points]]

# Generating notes

suggest similar versions

In [14]:
def extract_topics(text, n_topics=5):
    if not text or not text.strip():
        return []

    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=1000)
    X = vectorizer.fit_transform([text])

    words = vectorizer.get_feature_names_out()
    scores = X.toarray()[0]

    top_indices = np.argsort(scores)[::-1][:n_topics]
    return [words[i] for i in top_indices]

In [8]:
def generate_smart_notes(transcript, summary, topics):
    from nltk.tokenize import sent_tokenize

    sentences = sent_tokenize(transcript)

    notes = "\n📘 SMART NOTES\n"
    notes += "="*40 + "\n\n"

    notes += "🎯 Topic:\n"
    notes += ", ".join(topics[:3]) + "\n\n"

    notes += "🧠 Summary:\n"
    notes += summary.strip() + "\n\n"

    notes += "📌 Key Concepts:\n"
    important = sorted(sentences, key=lambda x: len(x), reverse=True)[:5]

    for i, s in enumerate(important, 1):
        notes += f"{i}. {s.strip()}\n"

    notes += "\n⚡ Quick Revision:\n"
    for t in topics:
        notes += f"- {t}\n"

    return notes

In [15]:
link = "https://youtu.be/CrEhGIBCAPU?si=XOKRH_qpxry0-K7-"
url = link
video_id = get_video_id(url)

try:
    transcript = get_transcript(video_id)
except Exception as e:
    print("Error: Transcript not available for this video.", e)
    transcript = ""

if not transcript:
    raise RuntimeError("Transcript extraction failed. Check video ID or availability.")

summary = summarize_text(transcript)
key_points = extract_key_points(transcript)
topics = extract_topics(transcript)
smart_notes = generate_smart_notes(transcript, summary, topics)

print(smart_notes)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


📘 SMART NOTES

🎯 Topic:
setback, people, think

🧠 Summary:
Perhaps if I could learn from it, I could understand it, and I could help others work through their next inevitable setback and come out the other side with a sense of resilience, creativity, and hopefully success. But the aftermath of a setback is when we come up with our best ideas. Establishing you're in a setback, not as intuitive as one may think. But some of us float unconsciously into our own setbacks without even realizing it or because we choose to ignore it. That I needed to show everyone that I was still worthy of existing in the workforce. Emerging from your setback is exhilarating. I am rebelliously optimistic that after today, you will see setbacks so much more clearly, and you will emerge from them with a sense of all you can achieve. Now ask yourself, how did that setback set the stage for your reinvention?

📌 Key Concepts:
1. So I spent three years
doing a deep dive into research, interviewing psychologists,
e